# 🧠 Intent Classification Layer

**Purpose**  
Layer ini bertujuan untuk **menyaring pesan teks** dan mengidentifikasi apakah pesan tersebut merupakan *progress report* atau bukan, sebelum diproses ke tahap ekstraksi informasi.


---

## ⚙️ Model Configuration

| Component | Method |
|----------|--------|
| Feature Extraction | TF-IDF |
| Classifier Models | Logistic Regression / Support Vector Machine |
| Library | scikit-learn |

---

## 🧩 Model Architecture

Pipeline yang digunakan:



Penjelasan:
- **TF-IDF (Term Frequency–Inverse Document Frequency)**  
  Mengubah teks menjadi representasi numerik berbasis bobot kata. TF-IDF : memperhitungkan seberapa sering kata muncul dalam sebuah dokumen dibandingkan dengan seluruh korpus, sehingga kata yang lebih “informatif” mendapat bobot lebih tinggi.

- **Classifier**  
  Random Forest : model ensemble berbasis decision tree yang bekerja dengan membangun banyak pohon keputusan (trees) dan menggabungkan hasil prediksi dari masing-masing pohon untuk menentukan output akhir. Model ini mampu menangkap pola yang lebih kompleks (non-linear) dalam data, karena setiap pohon membuat keputusan berdasarkan kombinasi fitur yang berbeda. Berbeda dengan Logistic Regression dan SVM yang bersifat linear, Random Forest mampu menangkap hubungan non-linear,

---

## 🎯 Output Specification

Model menghasilkan **binary classification**:

| Label | Description |
|------|------------|
| 1 | Progress (pesan laporan membaca Alkitab) |
| 0 | Non-progress (pesan biasa / noise) |

---

## 🔍 Functional Role in System

Layer ini memiliki peran penting dalam sistem:

- Menyaring pesan yang relevan  
- Mengurangi noise sebelum proses lanjutan  
- Meningkatkan akurasi ekstraksi (NER / parsing)

---

## 🔄 Processing Flow

| Step | Process | Output |
|------|--------|--------|
| 1 | Input raw text | Chat messages |
| 2 | TF-IDF transformation | Feature vectors |
| 3 | Classification | Random forest ML |

---

## 🧪 Evaluation Overview

Model dievaluasi menggunakan:

- Accuracy  
- Precision  
- Recall  
- F1-score  
- Confusion Matrix  

Tujuan evaluasi:
- Memastikan model mampu **mendeteksi progress message dengan baik**
- Meminimalkan kesalahan klasifikasi (false positive & false negative)

---

## ✨ Design Rationale

Pendekatan ini dipilih karena:

- TF-IDF efektif untuk representasi teks pendek seperti chat  
- Random forest, model ini bisa menangkap pola yang lebih kompleks dibanding model linear (SVM/Logistic regression)

---
## 0. Setup

In [ ]:
# System & Configuration
import warnings
import sys
from pathlib import Path
warnings.filterwarnings('ignore')
sys.path.append(str(Path('..').resolve() / 'src'))

import pandas as pd
# Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer
# Machine Learning Models 
from sklearn.ensemble import RandomForestClassifier
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
# Evaluation Metrics 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from config.settings import PROCESSED_DIR
plt.style.use('default')
sns.set()

# Data Directory 
CLS_DIR = PROCESSED_DIR / 'text classification'

print('Libraries and configuration loaded successfully.')

---
## 1. Load Data

In [ ]:
# ── Load Dataset dari CLS_DIR ──────────────────────────────────────────
train_df = pd.read_csv(CLS_DIR / 'train_classification.csv')
val_df   = pd.read_csv(CLS_DIR / 'val_classification.csv')
test_df  = pd.read_csv(CLS_DIR / 'test_classification.csv')

# ── Preview Data ───────────────────────────────────────────────────────
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

## 2. TF-IDF Vectorization

In [ ]:
# vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
vectorizer = TfidfVectorizer(max_features=50000, stop_words='english')
X_train = vectorizer.fit_transform(train_df['text'])
X_val = vectorizer.transform(val_df['text'])
X_test = vectorizer.transform(test_df['text'])

y_train = train_df['label']
y_val = val_df['label']
y_test = test_df['label']

---
## 3. Train Model (choose one)

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train
model.fit(X_train, y_train)

# Predict
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

In [ ]:
print("=== VALIDATION ===")
print("Accuracy:", accuracy_score(y_val, y_pred_val))
print(classification_report(y_val, y_pred_val))

print("\n=== TEST ===")
print("Accuracy:", accuracy_score(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))

---
## 4. Save Model

In [ ]:
# ── Save Model & Vectorizer (Joblib) ───────────────────────────────────
import joblib
from pathlib import Path

# pastikan folder ada
MODEL_DIR = CLS_DIR / 'model'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / 'random_forest_model.joblib'
VECTORIZER_PATH = MODEL_DIR / 'tfidf_vectorizer.joblib'

# save
joblib.dump(model, MODEL_PATH)
joblib.dump(vectorizer, VECTORIZER_PATH)

print("Model & vectorizer saved successfully using joblib.")

---
## 5. Try Classify DF

In [ ]:
# ── Load Model & Vectorizer ────────────────────────────────────────────
def load_model():
    model = joblib.load(CLS_DIR / 'model/random_forest_model.joblib')
    vectorizer = joblib.load(CLS_DIR / 'model/tfidf_vectorizer.joblib')
    return model, vectorizer

In [ ]:
def classify_dataframe(df, model, vectorizer):
    # pastikan kolom text tidak null
    df['text'] = df['text'].fillna('')

    # transform pakai TF-IDF 
    X = vectorizer.transform(df['text'])

    # prediksi random forest model
    df['prediction'] = model.predict(X)

    # ubah ke label yang readable
    df['label_name'] = df['prediction'].map({
        1: 'progress',
        0: 'non-progress'
    })
    return df

In [ ]:
df_test = pd.read_csv(CLS_DIR / 'test_classification.csv')
df_result = classify_dataframe(df_test, model, vectorizer)
df_result.head()